# 04 · Explainability (Grad-CAM, saliency) & computational efficiency
* **Grad-CAM** on the last conv block (`act5`), **saliency** = |∂P(fire)/∂input| (Eq. 16-19).
* **Efficiency** (Tables 4 & 7): parameters, model size, latency, GFLOPs. The paper's *0.95 s* inference time is ambiguous (per image? whole test set?); we report single-image latency **and** per-image time in a batch of 32. Measured FLOPs come out around 0.6 GFLOPs for FireNet-CNN, not the 2.085 stated in the paper.

In [ ]:
# ---- Colab setup (run first; if you ever see Keras-3 errors: Runtime > Restart session, then re-run) ----
import os, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"          # the paper's code style needs Keras 2
!pip -q install tf_keras
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/firenet"       # everything lives here, so a disconnect loses nothing
sys.path.insert(0, PROJECT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from types import SimpleNamespace
from IPython.display import Image, display
import firenet as fn                             # written to Drive by 00_setup_and_data.ipynb
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
DATASETS = ["A", "B"]          # which real dataset(s)
MODES    = ["paper", "strict"] # paper = split AFTER adding synthetic (as in the paper); strict = synthetic only in training folds
EPOCHS   = 2                   # <-- smoke test first! then set to 50 for the real run

cfg = SimpleNamespace(data=f"{PROJECT}/data", dataset="B", mode="paper", no_synth=False,
                      epochs=EPOCHS, bs=32, lr=1e-4, seed=42, out=f"{PROJECT}/runs")
def get(d):
    return fn.load_cached(cfg.data, d, use_synth=True, cache_dir=f"{PROJECT}/cache")

## Grad-CAM + saliency

In [ ]:
d, mode = DATASETS[-1], MODES[0]
model = tf.keras.models.load_model(f"{PROJECT}/runs/{d}_{mode}_firenet/model_fold1.keras")
X, y, s = get(d)
rng = np.random.default_rng(1)
real = ~s
idx = np.r_[rng.choice(np.where((y == 1) & real)[0], 4, replace=False), rng.choice(np.where((y == 0) & real)[0], 2, replace=False)]
fig, ax = plt.subplots(len(idx), 3, figsize=(9, 3 * len(idx)))
import cv2
for r, i in enumerate(idx):
    x = X[i].astype("float32") / 255.
    cam, p = fn.gradcam(model, x); sal = fn.saliency(model, x)
    heat = cv2.cvtColor(cv2.applyColorMap((cam * 255).astype("uint8"), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    ax[r, 0].imshow(X[i]); ax[r, 0].set_title(f"{'Fire' if y[i] else 'No fire'}  |  P(fire)={p:.3f}", fontsize=8)
    ax[r, 1].imshow(cv2.addWeighted(X[i], .6, heat, .4, 0)); ax[r, 1].set_title("Grad-CAM", fontsize=8)
    ax[r, 2].imshow(sal, cmap="hot"); ax[r, 2].set_title("Saliency map", fontsize=8)
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.show()
# NOTE: images may belong to fold-1's training set; for a fair qualitative check pick images from its validation fold.

### Does the model look at the fire - or at the background?
A quick quantitative check: how much Grad-CAM mass falls inside the HSV 'fire-colour' mask (a crude proxy, not ground truth).

In [ ]:
inside = []
for i in np.where((y == 1) & real)[0][:200]:
    x = X[i].astype("float32") / 255.; cam, _ = fn.gradcam(model, x)
    hsv = cv2.cvtColor(X[i], cv2.COLOR_RGB2HSV); m = cv2.inRange(hsv, (0, 100, 100), (20, 255, 255)) > 0
    if m.any() and cam.sum() > 0: inside.append(cam[m].sum() / cam.sum())
print(f"mean share of Grad-CAM mass on fire-coloured pixels: {np.mean(inside):.2f}  (fire-coloured pixels cover {m.mean():.2%} of the last image)")

## Efficiency (needs internet to fetch ImageNet weights for the baselines)

In [ ]:
fn.cmd_efficiency(SimpleNamespace(out=f"{PROJECT}/runs"))
pd.read_csv(f"{PROJECT}/runs/efficiency.csv")